# Deep Learning 081 — Masked Multi-Head Attention

Companion notebook to the lesson. One sentence carries the whole topic:

> **The transformer decoder is auto-regressive at inference time and
> non-auto-regressive at training time.**

This notebook measures why each half has to be true, and what it takes to make both true
of the same set of weights.

| Step | What we measure |
|---|---|
| the leak | change the LAST token, position 0 moves **0.243** |
| with the mask | position 0 moves **exactly 0.00** |
| softmax(-inf) | exactly 0, and rows still sum to 1 |
| the fill-value trap | −10 leaks **4.5e-05** per masked cell |
| prefix consistency | 1 pass vs 12 passes: **4.8e-07**. Without the mask: **0.79** |
| what it buys | **84×** at length 128 |
| what it costs | **49.99%** of the matrix discarded at n = 4096 |

Needs `torch` (CPU is fine). No training.

In [ ]:
import math, time
import torch
import torch.nn as nn

D, H = 128, 4

def causal_mask(n):
    '''0 on and below the diagonal, -inf strictly above it.'''
    return torch.full((n, n), float("-inf")).triu(1)

print(causal_mask(4))

That is the entire mechanism. It gets **added** to the scaled scores, **before** the softmax.

## Part A — Build masked attention

Exactly lesson 074's scaled dot-product attention, with one extra line.

In [ ]:
class MaskedAttention(nn.Module):
    def __init__(self, d=D, h=H):
        super().__init__()
        self.h, self.dk = h, d // h
        self.wq, self.wk = nn.Linear(d, d), nn.Linear(d, d)
        self.wv, self.wo = nn.Linear(d, d), nn.Linear(d, d)

    def forward(self, x, masked=True):
        n, d = x.shape
        split = lambda t: t.view(n, self.h, self.dk).transpose(0, 1)
        q, k, v = split(self.wq(x)), split(self.wk(x)), split(self.wv(x))
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        if masked:
            scores = scores + causal_mask(n)          # <- the whole difference
        w = torch.softmax(scores, dim=-1)
        return self.wo((w @ v).transpose(0, 1).reshape(n, d)), w

class DecoderBlock(nn.Module):
    def __init__(self, d=D):
        super().__init__()
        self.attn = MaskedAttention(d)
        self.ff = nn.Sequential(nn.Linear(d, 4*d), nn.ReLU(), nn.Linear(4*d, d))
        self.ln1, self.ln2 = nn.LayerNorm(d), nn.LayerNorm(d)

    def forward(self, x, masked=True):
        a, w = self.attn(x, masked=masked)
        x = self.ln1(x + a)
        x = self.ln2(x + self.ff(x))
        return x, w

## Part B — Is the leak real?

Take a sequence and change **only the last token** — a word that, at inference time, would
not have been generated yet when position 0 was produced. Then look at what moved.

Predict the two rows before running it.

In [ ]:
torch.manual_seed(1081)
blk = DecoderBlock()
x = torch.randn(6, D)
x2 = x.clone(); x2[-1] = torch.randn(D)          # only the FUTURE token changes

for masked in (False, True):
    with torch.no_grad():
        y1, _ = blk(x,  masked=masked)
        y2, _ = blk(x2, masked=masked)
    moved = (y1 - y2).abs().max(dim=1).values
    tag = "masked  " if masked else "unmasked"
    print(f"{tag} " + "  ".join(f"{v:.2e}" for v in moved))
print("\npositions 0-4 should be untouched; position 5 is the one we changed")

Unmasked, position 0's representation moved by **0.243** — it read a word that will not
exist when it matters. A model trained this way learns to depend on information it will
never have at inference, and the failure shows up only at prediction time, long after the
training loss started looking excellent.

Masked, the figure is not `1e-16`. It is **exactly 0.00**. The dependence is not reduced,
it is structurally absent: the future token's value never enters the sum.

## Part C — Why `-inf` and not "set the weights to zero afterwards"

In [ ]:
torch.manual_seed(2081)
w = torch.softmax(torch.randn(4, 4) + causal_mask(4), dim=-1)
print(w.round(decimals=4))
print(f"\nmax weight above the diagonal : {w.triu(1).abs().max():.1e}")
print(f"row sums                      : {w.sum(-1)}")
assert w.triu(1).abs().max() == 0.0
assert torch.allclose(w.sum(-1), torch.ones(4))

Adding **before** the softmax is what makes a renormalisation step unnecessary: the blocked
terms vanish from the numerator and the denominator together, so the survivors automatically
re-share the full 1.0. Note row 0 — with no past to attend to, the first token gives itself
a weight of exactly 1.

**Two traps that bite in practice.**

In [ ]:
# Trap 1 -- float16 caps at 65504, so people substitute a finite constant. Which ones work?
print(f"{'fill value':>12} {'stored as fp16':>16} {'leaked weight':>15}")
for fill in (float("-inf"), -1e9, -1e4, -100.0, -10.0):
    stored = torch.tensor(fill, dtype=torch.float16)
    s = torch.tensor([[0.0, stored.item()]], dtype=torch.float32)
    leaked = torch.softmax(s, dim=-1)[0, 1].item()
    name = "-inf" if fill == float("-inf") else f"{fill:g}"
    print(f"{name:>12} {stored.item():>16} {leaked:>15.2e}")

In [ ]:
# Trap 2 -- a row masked in EVERY column is 0/0
print("all -inf row ->", torch.softmax(torch.full((1, 3), float("-inf")), dim=-1))

`-inf`, `-1e9` and `-1e4` all leak exactly zero, but **−10 leaks 4.5e-05** on every masked
cell. And an entirely masked row gives `nan` — which is what happens when a padding mask and
a causal mask combine on a fully padded position, and it is the most common way a decoder
produces `nan` loss on step one.

## Part D — The identity that makes parallel training honest

Everything so far shows masking is *safe*. There is a stronger claim underneath, and it is
the one that actually justifies training one way and running another: the parallel
computation must produce **the same numbers** the sequential one would. If it were merely
close, the deployed model would not be the trained model.

In [ ]:
torch.manual_seed(3081)
blk = DecoderBlock()                                        # fresh block, matching the lesson
seq = torch.randn(12, D)

with torch.no_grad():
    parallel, _ = blk(seq)                                  # ONE pass, all positions
    stepwise = torch.stack([blk(seq[:t])[0][-1]             # twelve prefix passes
                            for t in range(1, 13)])

print(f"masked   : max |parallel - sequential| = {(parallel-stepwise).abs().max():.2e}")

with torch.no_grad():
    par_u, _ = blk(seq, masked=False)
    step_u = torch.stack([blk(seq[:t], masked=False)[0][-1] for t in range(1, 13)])
print(f"unmasked : max |parallel - sequential| = {(par_u-step_u).abs().max():.4f}")

**This is prefix consistency**, and it is the property masking really provides: a masked
decoder's output at position $t$ depends only on positions $0 \ldots t$, so feeding it a
prefix and feeding it the whole sequence agree at that position. That is why one pass can
stand in for $n$. Remove the mask and the gap is **0.79** — not an identity at all, which is
Part B's leak seen from the other side.

**Careful about what this does not fix.** The identity holds when the sequential pass is fed
the **ground-truth** prefix. At inference the model is fed its **own** previous outputs, which
will sometimes be wrong, and nothing here prepares it for that. The mismatch is called
**exposure bias**; masking makes no promise about it.

## Part E — What parallel training buys

In [ ]:
torch.manual_seed(4081)
print(f"{'length':>8} {'sequential':>13} {'parallel':>11} {'speedup':>10}")
for n in (16, 32, 64, 128):
    seq = torch.randn(n, D)
    with torch.no_grad():
        t0 = time.perf_counter()
        for _ in range(3):
            for t in range(1, n + 1):
                blk(seq[:t])
        t_seq = (time.perf_counter() - t0) / 3

        t0 = time.perf_counter()
        for _ in range(3):
            blk(seq)
        t_par = (time.perf_counter() - t0) / 3
    print(f"{n:>8} {t_seq:>12.4f}s {t_par:>10.4f}s {t_seq/t_par:>9.1f}x")

Your absolute times will differ; the **ratio** is the point, and it grows with length because
the sequential version pays $n$ forward passes while the parallel one pays 1. At the sequence
lengths real corpora use, this is the difference between a trainable model and an untrainable
one.

## Part F — What masking costs

In [ ]:
print(f"{'n':>8} {'computed':>14} {'kept':>12} {'discarded':>11}")
for n in (8, 64, 512, 4096):
    total, kept = n * n, n * (n + 1) // 2
    print(f"{n:>8} {total:>14,} {kept:>12,} {100*(total-kept)/total:>10.2f}%")

The discarded share tends to $(n-1)/2n$, just under half — and in this implementation those
scores are **computed and then thrown away**, because the mask is added to a full $n \times n$
matrix that has already been formed. **Masking buys correctness, not speed.** The speedup in
Part E came from processing positions in parallel, not from the mask. (Fused kernels such as
FlashAttention's causal mode do skip the upper blocks; that is an implementation gain, and it
is reported here rather than measured.)

## What to take away

- **Auto-regressive at inference, non-auto-regressive at training.** Inference has no choice;
  training does, because the target sentence is already known.
- **Unmasked self-attention over the target leaks** — position 0 moves 0.243 when only the
  last token changes. Masked, it moves **exactly 0.00**.
- **The mask is $-\infty$ added to the upper triangle before the softmax.** No renormalisation
  is needed; row 0 is exactly 1.0 on itself.
- **Prefix consistency is the real property:** 1 pass equals 12 passes to 4.8e-07, and the
  claim is false (0.79) without the mask. That is what makes parallel training legitimate.
- **It does not fix exposure bias.**

**Exercises**

1. Build the mask with `torch.tril` instead of `triu` and multiply the weights *after* the
   softmax, renormalising by hand. Confirm you get the same answer — then say why the
   `-inf` version is preferred anyway.
2. Stack six `DecoderBlock`s and re-run Part D. Does prefix consistency survive depth? Should
   it?
3. Combine a causal mask with a padding mask for a batch containing one fully padded row.
   Reproduce the `nan`, then fix it.
4. Replace `-inf` with `-10` in `causal_mask` and re-run Part B. How large does the leak at
   position 0 become?